In [14]:
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext

# # Define arguments
# args = getResolvedOptions(sys.argv, ['JOB_NAME'])
JOB_NAME="job_bookmark_test"


In [15]:
# Initialize Spark and Glue contexts
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session


ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=pyspark-shell, master=local[*]) created by __init__ at /tmp/ipykernel_128/3743495615.py:2 

In [29]:
job = Job(glueContext)
job.init(JOB_NAME)

Exception while setting endpoint config: java.lang.NullPointerException


25/03/17 15:39:08 WARN Job$: Job run ID job_bookmark_test is either null or empty or its same as Job name. 


In [30]:
spark

In [31]:
input_path = "s3://edp-source/source_files/"
output_path = "s3://edp-source/incremental_data"

In [32]:
# Read data using Glue's DynamicFrame with job bookmarks
datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={
        "paths": [input_path],
        "recurse": True,
        "groupFiles": "none"
    },
    format="csv",
    format_options={"withHeader": True}
)

In [33]:
datasource.show(5)

{"order_id": "1", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11599", "order_status": "CLOSED"}
{"order_id": "2", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "256", "order_status": "PENDING_PAYMENT"}
{"order_id": "3", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "12111", "order_status": "COMPLETE"}
{"order_id": "4", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "8827", "order_status": "CLOSED"}
{"order_id": "5", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11318", "order_status": "COMPLETE"}


In [34]:
# Write back the data to the output S3 path
glueContext.write_dynamic_frame.from_options(
    frame=datasource,
    connection_type="s3",
    connection_options={
        "path": output_path,
        "partitionKeys": []
    },
    format="csv"
)

In [35]:
# Commit the job with job bookmark
job.commit()

25/03/17 16:54:38 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 4431515 ms exceeds timeout 120000 ms
25/03/17 16:54:38 WARN SparkContext: Killing executors is not supported by current scheduler.
